# **Class Project U3 — Markov Decision Processes**
**Master's in Computer Science — Reinforcement Learning - Ana Sofía Rentería Palomares** 

**Problem Statement**

Can-collecting robot in an office environment. The agent decides how to act based on the battery charge level.

- **States:** `high`, `low`
- **Actions:** `search`, `wait` (from `high`); `search`, `wait`, `recharge` (from `low`)
- **Parameters:** α (prob. of keeping battery high when searching from high), β (prob. of keeping battery low when searching from low)


## Task 1 — Simulation of MDP Transition Dynamics

Write a Python script to simulate the transition dynamics of the can-collecting robot described in the
provided materials. Your implementation must follow the theoretical model, including the defined states, actions,
rewards, and transition probabilities. Assign values to the parameters 𝜶 and 𝜷, and use them to model the
stochastic behavior of the system. Run multiple simulations and analyze the impact of different values of 𝜶 and
𝜷 on the robot's behavior and outcomes.

In [1]:
import random                          
import numpy as np                     
import matplotlib.pyplot as plt        
import plotly.graph_objects as go     
from collections import defaultdict    

### Global Parameters and `build_dynamics` Function

#### Model Parameters

- **`ALPHA` (α):** Probability that the battery stays at `high` after performing `search` from `high`. With α=0.7, the robot searches 70% of the time without draining the battery.
- **`BETA` (β):** Probability that the battery stays at `low` after performing `search` from `low`. With β=0.5, there is a 50% risk of running out of battery (and being rescued).
- **`R_SEARCH`:** Expected reward for searching.
- **`R_WAIT`:** Expected reward for waiting.
- **`R_RESCUE`:** Penalty when the battery runs out and the robot must be rescued.

#### `build_dynamics` Function

Builds the **MDP dynamics dictionary**, which represents the transition function `p(s', r | s, a)`. This dictionary describes all possible consequences of each action in each state.

**Key structure:** `(current_state, action, next_state, reward)` ----> probability


| (s, a, s', r) | Probability | Meaning |
|---|---|---|
| (high, search, high, r_search) | α | Searches from high and battery stays high |
| (high, search, low, r_search) | 1−α | Searches from high and battery drops to low |
| (high, wait, high, r_wait) | 1 | Waiting from high always keeps battery high |
| (low, search, low, r_search) | β | Searches from low and battery stays low |
| (low, search, high, r_rescue) | 1−β | Battery runs out so robot must be rescued (recharges to high), reward is negative |
| (low, wait, low, r_wait) | 1 | Waiting from low never changes state, stays at low |
| (low, recharge, high, 0) | 1 | Recharging always leads to high, with no positive reward|

In [2]:
# Global MDP parameters 
ALPHA = 0.7   # α: prob. of keeping battery high when searching from high
BETA  = 0.5   # β: prob. of keeping battery low  when searching from low

R_SEARCH = 3   # Reward for searching 
R_WAIT   = 1   # Reward for waiting 
R_RESCUE = -3  # Penalty: robot runs out of battery and must be rescued


def build_dynamics(alpha, beta, r_search=R_SEARCH, r_wait=R_WAIT, r_rescue=R_RESCUE):
    """
    Builds and returns the MDP dynamics dictionary.

    The dictionary represents p(s', r | s, a)
    """
    dynamics = {
        # State 'high', action 'search' 
        # With probability α: battery stays high and cans are collected
        ("high", "search", "high", r_search): alpha,
        # With probability 1-α: battery drops to 'low' (but cans are still collected)
        ("high", "search", "low",  r_search): 1 - alpha,

        # State 'high', action 'wait' 
        # Waiting always keeps battery high (probability 1)
        ("high", "wait", "high", r_wait): 1,

        # State 'low', action 'search' 
        # With probability β: battery stays low and cans are collected
        ("low", "search", "low",  r_search): beta,
        # With probability 1-β: battery runs out → rescue (-3) → recharge to high
        ("low", "search", "high", r_rescue): 1 - beta,

        # State 'low', action 'wait' 
        # Waiting from 'low' always keeps battery at 'low'
        ("low", "wait", "low", r_wait): 1,

        # State 'low', action 'recharge' 
        # Recharging always leads to state 'high' with reward 0
        # (robot goes to base, does not collect cans during that time)
        ("low", "recharge", "high", 0): 1,
    }
    return dynamics

### Simulation Functions: `simulate_step` and `run_episode`

#### `simulate_step(state, action, dynamics)`
Simulates a single MDP step. Given a state and an action, it searches the `dynamics` dictionary for all possible transitions and randomly selects one according to the defined probabilities.

- Uses `random.choices(transitions, weights=probabilities)` for stochastic sampling. This directly implements the equation `p(s', r | s, a)`.
- Returns a tuple `(next_state, reward)`.

#### `run_episode(policy, dynamics, steps, initial_state)`
Executes a **complete episode** of `steps` steps applying a given policy.

- **`policy`:** function that receives the current state and returns the action to take.
- At each step: query the policy → simulate the step → accumulate the reward.
- Returns:
  - `history`: list of tuples `(state, action, next_state, reward)` for each step.
  - `cumulative_rewards`: list where element `i` is the total sum of rewards up to step `i`.

In [3]:
def simulate_step(state, action, dynamics):
    """
    Simulates a single MDP step.

    Searches the 'dynamics' dictionary for all entries matching
    the current state and action, then samples the outcome
    according to the transition probabilities.

    Returns: (next_state, reward)
    """
    transitions   = []   # List of possible (next_state, reward) outcomes
    probabilities = []   # Probability corresponding to each transition

    # Iterate over all entries in the dynamics dictionary
    for (s, a, s_next, r), prob in dynamics.items():
        # Only consider transitions matching the current state and action
        if s == state and a == action:
            transitions.append((s_next, r))
            probabilities.append(prob)

    # random.choices samples 1 outcome using probabilities as weights
    # This directly implements the sampling of p(s', r | s, a)
    return random.choices(transitions, weights=probabilities, k=1)[0] # Returns: (next_state, reward)


def run_episode(policy, dynamics, steps=100, initial_state="high"):
    """
    Executes a complete episode of 'steps' steps.
      1. The policy chooses an action based on the current state.
      2. simulate_step determines the outcome (next_state, reward).
      3. Reward is accumulated and the next state is advanced.

    Returns:
      history: list of (state, action, next_state, reward) per step
      cumulative_rewards: stores the cumulative reward for each step
    """
    state   = initial_state  # Episode starting state
    history = []             # Record of each step 
    cumulative         = 0   # Running sum of rewards
    cumulative_rewards = []  # Stores the cumulative total for each step

    for _ in range(steps):
        action              = policy(state)                        # Policy decides which action to take
        next_state, reward  = simulate_step(state, action, dynamics) 
        cumulative         += reward                              
        history.append((state, action, next_state, reward))       
        cumulative_rewards.append(cumulative)                      
        state = next_state                                         # Advance to next state

    return history, cumulative_rewards

###  Transition Demonstration


 **Runs a 10-step episode** with the always_search policy to show how the robot transitions between states and what rewards it receives.

In [4]:
# Always-search policy 
# Ignores the current state and always returns 'search'.
def policy_always_search(state):
    return "search"

# Build the dictionary with values α=0.7, β=0.5
dynamics_demo = build_dynamics(ALPHA, BETA)

# Print the transition table 

print(f"MDP Dynamics  (α={ALPHA}, β={BETA})")
print(f"{'(s, a, s\', r)':<40} {'p':>6}")
for (s, a, s_next, r), prob in dynamics_demo.items():
    print(f"({s}, {a}, {s_next}, {r}){'':<{35 - len(s+a+s_next+str(r))}} {prob:>6.2f}")

# Simulate 10 steps and print the step history

print("10-step simulation with policy: always_search")
history, _ = run_episode(policy_always_search, dynamics_demo, steps=10)
print(f"{'Step':<6} {'State':<8} {'Action':<10} {'Next State':<14} {'Reward'}")
print("-" * 50)
for i, (s, a, s_next, r) in enumerate(history, 1):
    # enumerate(history, 1) starts the counter at 1 instead of 0
    print(f"{i:<6} {s:<8} {a:<10} {s_next:<14} {r}")

MDP Dynamics  (α=0.7, β=0.5)
(s, a, s', r)                                 p
(high, search, high, 3)                       0.70
(high, search, low, 3)                        0.30
(high, wait, high, 1)                         1.00
(low, search, low, 3)                         0.50
(low, search, high, -3)                       0.50
(low, wait, low, 1)                           1.00
(low, recharge, high, 0)                      1.00
10-step simulation with policy: always_search
Step   State    Action     Next State     Reward
--------------------------------------------------
1      high     search     low            3
2      low      search     high           -3
3      high     search     high           3
4      high     search     high           3
5      high     search     high           3
6      high     search     high           3
7      high     search     high           3
8      high     search     high           3
9      high     search     high           3
10     high     search 

###  Analysis of the Impact of α and β on Reward

Here we quantitatively evaluate how the robot's performance changes when the battery reliability parameters are altered.

Three (α, β) configurations are defined: high reliability, moderate, and low reliability.
- For each configuration, 50 episodes of 200 steps each are run.
- The average cumulative reward at the end of the 200 steps is printed to compare configurations.

In [5]:
# Number of steps per episode and episodes per configuration
STEPS    = 200
EPISODES = 50

# Each combo defines α, β, and a descriptive label
param_combos = [
    (0.9, 0.8, "α=0.9, β=0.8 (high reliability)"),   # Very reliable battery
    (0.6, 0.6, "α=0.6, β=0.6 (moderate)"),             # Reliable battery
    (0.4, 0.3, "α=0.4, β=0.3 (low reliability)"),    # Very unreliable battery
]

print(f"{'Configuration':<35} {'Average Cumulative Reward':>30}")
for alpha, beta, label in param_combos:
    dyn = build_dynamics(alpha, beta)  # Build dynamics with these parameters
    totals = []                        # Store the final reward of each episode

    for _ in range(EPISODES):
        _, cum = run_episode(policy_always_search, dyn, steps=STEPS)
        totals.append(cum[-1])  # cum[-1] is the cumulative reward at the end of the episode

    # np.mean computes the average over the 50 episodes
    print(f"{label:<35} {np.mean(totals):>30.2f}")

Configuration                            Average Cumulative Reward
α=0.9, β=0.8 (high reliability)                             523.44
α=0.6, β=0.6 (moderate)                                     356.76
α=0.4, β=0.3 (low reliability)                              214.32



## Task 2 — System Behavior Visualizations

Using the simulation developed in Task 1, generate visualizations to analyze the behavior of the system.

•Plot the frequency of each state.

• The accumulated reward over time,

• A comparison of the system's performance for different values of 𝜶 and 𝜷.

• Based on your results, discuss how these parameters influence the dynamics and overall performance of the robot.

###  Figure 1: State Frequency

A 500-step episode is run and the number of times the robot was in each state (high or low) is counted. 

- If high dominates: the policy keeps the battery charged most of the time.
- If low dominates: the robot maintains or transitions to low battery, which increases risk.

In [6]:
# Build the main dynamics with global ALPHA and BETA values
dynamics_main = build_dynamics(ALPHA, BETA)


history_long, cum_long = run_episode(policy_always_search, dynamics_main, steps=500)

# Extract only the state column from the history
states_visited = [s for s, _, _, _ in history_long]
freq_high = states_visited.count('high')
freq_low  = states_visited.count('low')

fig = go.Figure()

fig.add_trace(go.Bar(
    x=['high', 'low'],
    y=[freq_high, freq_low],
    marker_color=['steelblue', 'salmon'],
    customdata=[
        [ALPHA, BETA, freq_high, round(100*freq_high/500, 1)],
        [ALPHA, BETA, freq_low,  round(100*freq_low/500, 1)],
    ],
    hovertemplate=(
        '<b>State: %{x}</b><br>'
        'Frequency: %{customdata[2]} times<br>'
        'Percentage: %{customdata[3]}%<br>'
        'α = %{customdata[0]}<br>'
        'β = %{customdata[1]}'
        '<extra></extra>'
    ),
    text=[f"{freq_high}<br>({100*freq_high/500:.1f}%)", f"{freq_low}<br>({100*freq_low/500:.1f}%)"],
    textposition='outside'
))

fig.update_layout(
    title=f'State Frequency (α={ALPHA}, β={BETA}, 500 steps)',
    xaxis_title='State',
    yaxis_title='Frequency',
    yaxis_range=[0, 550],
    width=500, height=420,
    template='plotly_white'
)
fig.show()

print(f"State 'high': {freq_high} times ({100*freq_high/500:.1f}%)")
print(f"State 'low':  {freq_low} times ({100*freq_low/500:.1f}%)")

State 'high': 308 times (61.6%)
State 'low':  192 times (38.4%)


###  Figure 2: Cumulative Reward Over Time

Plots the **sum of rewards obtained up to each step** of the long 500-step episode. This curve shows:

- The **overall trend** (positive slope = the robot earns more than it loses).
- **Drops** caused by rescue penalties (−3), visible as sharp downward dips in the curve.
- The **variability** of the stochastic process.

In [7]:
# Reuse cum_long and history_long computed in the previous cell
steps_range       = list(range(1, 501))
rewards_per_step  = [r for _, _, _, r in history_long]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=steps_range,
    y=cum_long,
    mode='lines',
    line=dict(color='steelblue', width=1.8),
    customdata=list(zip(rewards_per_step, [ALPHA]*500, [BETA]*500)),
    hovertemplate=(
        '<b>Step: %{x}</b><br>'
        'Cumulative reward: %{y}<br>'
        'Reward at this step: %{customdata[0]}<br>'
        'α = %{customdata[1]}<br>'
        'β = %{customdata[2]}'
        '<extra></extra>'
    ),
    name='Cumulative reward'
))

fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=1)

fig.update_layout(
    title=f'Cumulative Reward (α={ALPHA}, β={BETA}, policy: always_search)',
    xaxis_title='Step',
    yaxis_title='Cumulative reward',
    width=800, height=420,
    template='plotly_white'
)
fig.show()

###  Figure 3: Performance Comparison for Different α and β Values

To compare the effect of the parameters, 30 episodes of 300 steps per configuration are averaged. The plot shows:

- Solid line: average cumulative reward over time.
- Semi-transparent band: standard deviation, representing variability across episodes.

The higher the curve and the narrower the band, the better and more stable that configuration is.

In [8]:
STEPS_VIZ    = 300
EPISODES_VIZ = 30

configs = [
    (0.9, 0.8, 'α=0.9, β=0.8', 'green'),
    (0.7, 0.5, 'α=0.7, β=0.5', 'steelblue'),
    (0.4, 0.3, 'α=0.4, β=0.3', 'tomato'),
    (0.5, 0.9, 'α=0.5, β=0.9', 'orange'),
]

fig = go.Figure()
steps_x = list(range(1, STEPS_VIZ + 1))

for alpha, beta, label, color in configs:
    dyn     = build_dynamics(alpha, beta)
    all_cum = []
    for _ in range(EPISODES_VIZ):
        _, cum = run_episode(policy_always_search, dyn, steps=STEPS_VIZ)
        all_cum.append(cum)
    mean_cum = np.mean(all_cum, axis=0)
    std_cum  = np.std(all_cum, axis=0)

    # Banda de desviación estándar
    fig.add_trace(go.Scatter(
        x=steps_x + steps_x[::-1],
        y=list(mean_cum + std_cum) + list((mean_cum - std_cum)[::-1]),
        fill='toself',
        fillcolor=color,
        opacity=0.15,
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip'
    ))

    # Mean line with tooltip
    fig.add_trace(go.Scatter(
        x=steps_x,
        y=mean_cum,
        mode='lines',
        name=label,
        line=dict(color=color, width=2),
        customdata=list(zip(mean_cum.round(2), std_cum.round(2), [alpha]*STEPS_VIZ, [beta]*STEPS_VIZ)),
        hovertemplate=(
            '<b>Step: %{x}</b><br>'
            'Average reward: %{customdata[0]}<br>'
            'Std. deviation: ±%{customdata[1]}<br>'
            'α = %{customdata[2]}<br>'
            'β = %{customdata[3]}'
            '<extra></extra>'
        )
    ))

fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=1)

fig.update_layout(
    title='Performance Comparison for Different α and β Values (policy: always_search)',
    xaxis_title='Step',
    yaxis_title='Average cumulative reward',
    width=900, height=480,
    template='plotly_white',
    legend=dict(title='Configuration')
)
fig.show()

### Conclusions on the Impact of α and β on the Robot's Dynamics

- **High α** (e.g. 0.9): the robot stays in the `high` state more often when searching, allowing the `search` action to be exploited continuously without constantly dropping to `low`. The result is a higher cumulative reward with lower variance.

- **High β** (e.g. 0.9): when the robot is already in `low`, the battery is less likely to run out while searching, so it can keep collecting cans without being rescued. This improves performance in the `low` state, even though the robot is already in an unfavorable situation.

- **Low α and β** (e.g. 0.4 and 0.3): the robot discharges more frequently, accumulates rescue penalties (−3), and performance drops significantly. Cumulative reward can turn negative in long episodes.

- **Parameter interaction:** even with high β, if α is low the robot will constantly drop to `low`. Therefore, α has a greater impact on overall performance than β.


## Task 3 — Policy Definition and Evaluation

Define and evaluate different policies for the robot. A policy specifies which action the agent takes in
each state. Implement at least two different policies for comparison. Examples of policies: always search (greedy
but risky), always wait (safe but non-greedy), or balanced strategy (when in low, recharge). For each policy, run
multiple simulations (at least 50 episodes per policy) and compute the average accumulated reward. Compare the
results and discuss which policy performs better and why.

###  Policy Definitions



| Policy | In state `high` | In state `low` | Strategy |
|---|---|---|---|
| `always_search` | search | search | Maximum reward, maximum risk |
| `always_wait` | wait | wait | No risk, minimum reward |
| `balanced` | search | recharge | Exploits `high`, avoids rescue from `low` |
| `adaptive` | search | wait | Exploits `high`, earns some reward from `low` without risk |

In [9]:
# Policy 1: Always Search (greedy) 
# Always searches for cans regardless of battery level.
# Maximizes short-term reward, but risks frequent rescues from the 'low' state.
def policy_always_search(state):
    return "search"


# Policy 2: Always Wait (conservative) 
# Always waits. No rescues or penalties ever occur, but the reward
# is minimal (r_wait = 1) and battery level never changes.
def policy_always_wait(state):
    return "wait"


# Policy 3: Balanced 
# Searches when battery is high (no immediate rescue risk)
# and recharges when low (completely avoids depletion risk).
# Trade-off: recharge steps generate no reward.
def policy_balanced(state):
    if state == "high":
        return "search"    # Worth searching: there is battery margin
    else:
        return "recharge"  # Prioritizes recovering battery over collecting cans


# Policy 4: Adaptive 
# Searches in 'high' and waits in 'low'.
# Unlike 'balanced', no steps are spent recharging:
# in 'low' it still earns r_wait = 1 with no risk at all.
def policy_adaptive(state):
    if state == "high":
        return "search"
    else:
        return "wait"   # Safe and produces some reward


# Dictionary grouping all policies for easy iteration
policies = {
    "always_search": policy_always_search,
    "always_wait":   policy_always_wait,
    "balanced":      policy_balanced,
    "adaptive":      policy_adaptive,
}

###  Statistical Evaluation: 50 Episodes per Policy

To compare policies, 50 episodes of 200 steps are run for each one. The following metrics are computed:

- Mean: expected average performance.
- Std: how much performance varies across episodes (variability/risk).
- Min / Max: worst and best observed case.

The complete curves for each episode are also stored in `all_curves` for use in the plots.

In [10]:
NUM_EPISODES  = 50    # Episodes per policy (minimum required by the project)
EPISODE_STEPS = 200   # Steps per episode

# Build dynamics with the global parameters
dynamics_eval = build_dynamics(ALPHA, BETA)

results    = {}   # {policy_name: [final_reward_episode_1, ...]}
all_curves = {}   # {policy_name: array of shape (50, 200)}

for name, policy in policies.items():
    finals = []   # Cumulative reward at the end of each episode
    curves = []   # Full cumulative reward curve per episode

    for _ in range(NUM_EPISODES):
        _, cum = run_episode(policy, dynamics_eval, steps=EPISODE_STEPS)
        finals.append(cum[-1])  # Only the final value is stored for the summary table
        curves.append(cum)      # Full curve stored for plotting

    results[name]    = finals
    all_curves[name] = np.array(curves)  # Convert to array to compute mean/std per column

# Summary table 
print(f"{'Policy':<20} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 62)
for name, finals in results.items():
    arr = np.array(finals)
    print(f"{name:<20} {arr.mean():>10.2f} {arr.std():>10.2f} {arr.min():>10.2f} {arr.max():>10.2f}")

Policy                     Mean        Std        Min        Max
--------------------------------------------------------------
always_search            374.40      21.70     330.00     432.00
always_wait              200.00       0.00     200.00     200.00
balanced                 460.50      13.69     432.00     495.00
adaptive                 207.76       6.41     202.00     228.00


### Figure 4: Average Cumulative Reward Curves per Policy

Plots the temporal evolution of the cumulative reward for each policy, averaged over 50 episodes.
This allows us to see not only the final value, but also how quickly the reward grows and how stable each policy is over time.

In [11]:
colors_pol = {
    'always_search': 'tomato',
    'always_wait':   'steelblue',
    'balanced':      'green',
    'adaptive':      'orange',
}

fig = go.Figure()
steps_x = list(range(1, EPISODE_STEPS + 1))

for name, curves in all_curves.items():
    mean_c = curves.mean(axis=0)
    std_c  = curves.std(axis=0)
    color  = colors_pol[name]

    # Banda de desviación estándar
    fig.add_trace(go.Scatter(
        x=steps_x + steps_x[::-1],
        y=list(mean_c + std_c) + list((mean_c - std_c)[::-1]),
        fill='toself',
        fillcolor=color,
        opacity=0.12,
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip'
    ))

    # Mean line with tooltip
    fig.add_trace(go.Scatter(
        x=steps_x,
        y=mean_c,
        mode='lines',
        name=name,
        line=dict(color=color, width=2),
        customdata=list(zip(mean_c.round(2), std_c.round(2), [ALPHA]*EPISODE_STEPS, [BETA]*EPISODE_STEPS)),
        hovertemplate=(
            '<b>Step: %{x}</b><br>'
            'Average reward: %{customdata[0]}<br>'
            'Std. deviation: ±%{customdata[1]}<br>'
            'α = %{customdata[2]}<br>'
            'β = %{customdata[3]}'
            '<extra></extra>'
        )
    ))

fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=1)

fig.update_layout(
    title=f'Average Cumulative Reward per Policy (α={ALPHA}, β={BETA}, {NUM_EPISODES} episodes)',
    xaxis_title='Step',
    yaxis_title='Average cumulative reward',
    width=900, height=480,
    template='plotly_white',
    legend=dict(title='Policy')
)
fig.show()

### Figure 5: Boxplot of Final Reward per Policy

The boxplot shows the complete distribution of cumulative reward at the end of the 50 episodes for each policy:



In [12]:
colors_pol = {
    'always_search': 'tomato',
    'always_wait':   'steelblue',
    'balanced':      'green',
    'adaptive':      'orange',
}

fig = go.Figure()

for name in policies:
    fig.add_trace(go.Box(
        y=results[name],
        name=name,
        marker_color=colors_pol[name],
        boxmean=True,
        customdata=[[ALPHA, BETA]] * NUM_EPISODES,
        hovertemplate=(
            '<b>Policy: ' + name + '</b><br>'
            'Final reward: %{y}<br>'
            'α = %{customdata[0]}<br>'
            'β = %{customdata[1]}'
            '<extra></extra>'
        )
    ))

fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=1)

fig.update_layout(
    title=f'Final Reward Distribution — {NUM_EPISODES} episodes per policy',
    xaxis_title='Policy',
    yaxis_title=f'Cumulative reward ({EPISODE_STEPS} steps)',
    width=700, height=480,
    template='plotly_white'
)
fig.show()

### Discussion — Policy Comparison

| Policy | Description | Expected Behavior |
|---|---|---|
| `always_search` | Always searches regardless of battery level | High reward but high variance due to frequent rescues from `low` |
| `always_wait` | Always waits | Low but constant reward; no rescues or large losses ever occur |
| `balanced` | Searches in `high`, recharges in `low` | Avoids rescue risk; may sacrifice reward by recharging frequently |
| `adaptive` | Searches in `high`, waits in `low` | Compromise: earns some reward in `low` without rescue risk |

**Conclusions:**

- With **high α and β**, `always_search` tends to be the best policy because the probability of depletion is low, so the rescue penalty rarely triggers and can collection is maximized.

- With **low α and β**, `always_search` can become the worst policy, as rescue penalties accumulate. In that scenario, `balanced` or `adaptive` outperform the greedy policy by avoiding rescue costs.

- `always_wait` is always suboptimal in terms of absolute reward, but it is the most predictable and never produces negative rewards.

- The `balanced` policy is a robust strategy that works well across a wide range of α and β values, and is generally the best choice when there is uncertainty about the environment's parameters.